# Klasifikasi Kacamata dan Helm Menggunakan HOG + PCA + KNN
Notebook ini adalah versi implementasi klasifikasi menggunakan metode ekstraksi fitur HOG (Histogram of Oriented Gradients), reduksi dimensi PCA, dan model **K-Nearest Neighbors (KNN)**.

In [ ]:
import cv2
import numpy as np
import os
import glob
from tqdm.notebook import tqdm
from skimage.feature import hog

IMG_SIZE = (64, 64)

def extract_features_hog(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None

    # Resize dan konversi ke Grayscale
    img_resized = cv2.resize(img, IMG_SIZE)
    img_gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    
    # Ekstraksi fitur HOG (Histogram of Oriented Gradients)
    # Menggunakan parameter standar untuk deteksi kontur/bentuk objek
    features = hog(img_gray, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)
    return features

train_dir = "dataset/train"
categories = {"helm": 0, "kacamata": 1}

X_train_list = []
y_train_list = []

for label_name, label_id in categories.items():
    folder = os.path.join(train_dir, label_name)
    files = glob.glob(os.path.join(folder, "*.*"))
    print(f"Memproses '{label_name}': {len(files)} gambar ditemukan...")

    for img_path in tqdm(files, desc=f"Ekstraksi {label_name}"):
        features = extract_features_hog(img_path)
        if features is not None:
            X_train_list.append(features)
            y_train_list.append(label_id)

X = np.array(X_train_list)
y = np.array(y_train_list)

print(f"\nEkstraksi selesai! Dimensi Matriks Fitur Asli (X): {X.shape}")


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

print("Standarisasi dan PCA...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Reduksi dimensi menjadi 100 komponen utama untuk menghindari overfitting
pca = PCA(n_components=100, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Dimensi Matriks Fitur setelah PCA: {X_pca.shape}")

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier

X_tr, X_val, y_tr, y_val = train_test_split(X_pca, y, test_size=0.2, random_state=42, stratify=y)

print("Melatih model K-Nearest Neighbors (KNN) dengan PCA + HOG...")
# Menggunakan k=5 dengan metrik jarak Minkowski (Euclidean jika p=2)
knn_model = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
knn_model.fit(X_tr, y_tr)

val_acc = knn_model.score(X_val, y_val)
print(f"Akurasi pada data validasi internal (20%): {val_acc*100:.2f}%")

cv_scores = cross_val_score(knn_model, X_pca, y, cv=5, scoring='accuracy', n_jobs=-1)
print(f"5-Fold CV Scores: {[round(s*100, 2) for s in cv_scores]}")
print(f"Rata-rata CV Accuracy: {cv_scores.mean()*100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt 

def classify_and_visualize_hog(image_path, model, pca_model, scaler_model):
    img = cv2.imread(image_path)
    if img is None:
        return "Error"

    img_resized = cv2.resize(img, IMG_SIZE)
    img_gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    
    features, hog_image = hog(img_gray, orientations=9, pixels_per_cell=(8, 8),
                              cells_per_block=(2, 2), block_norm='L2-Hys', visualize=True)

    features_scaled = scaler_model.transform([features])
    features_pca = pca_model.transform(features_scaled)
    pred_id = model.predict(features_pca)[0]
    predicted_label = "Helm" if pred_id == 0 else "Kacamata"

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)); axes[0].set_title("1. Original (Resized)")
    axes[1].imshow(img_gray, cmap="gray"); axes[1].set_title("2. Grayscale")
    axes[2].imshow(hog_image, cmap="gray"); axes[2].set_title(f"3. Prediksi KNN: {predicted_label}")
    
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    return predicted_label

print("=== DEMO KLASIFIKASI ===")
helm_samples = glob.glob("dataset/test/helm/*.*")
if helm_samples:
    classify_and_visualize_hog(helm_samples[0], knn_model, pca, scaler)

kaca_samples = glob.glob("dataset/test/kacamata/*.*")
if kaca_samples:
    classify_and_visualize_hog(kaca_samples[0], knn_model, pca, scaler)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

test_dir = "dataset/test"
test_files = glob.glob(os.path.join(test_dir, "*", "*.*"))

y_true = []
y_pred = []

print(f"Memproses {len(test_files)} gambar data test...")

for img_path in tqdm(test_files, desc="Evaluasi Test Data"):
    parent = os.path.basename(os.path.dirname(img_path))
    actual = "Helm" if parent.lower() == "helm" else "Kacamata"
    y_true.append(actual)
    
    img = cv2.imread(img_path)
    if img is not None:
        features = extract_features_hog(img_path)
        features_scaled = scaler.transform([features])
        features_pca = pca.transform(features_scaled)
        
        pred_id = knn_model.predict(features_pca)[0]
        pred = "Helm" if pred_id == 0 else "Kacamata"
        y_pred.append(pred)
    else:
        y_pred.append("Error")

labels = ["Helm", "Kacamata"]
cm = confusion_matrix(y_true, y_pred, labels=labels)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - HOG + PCA + KNN")
plt.show()

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
